Objective

Import, Settings and Paths

In [ ]:
import os
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

for p in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

Load Raw Data

In [ ]:
RAW_FILE = DATA_RAW / "statcan_14-10-0416-01_full.csv"

print("Raw dataset path:")
print(RAW_FILE)

Initial Validation

In [ ]:
# Read a small sample to inspect the dataset structure
sample = pd.read_csv(RAW_FILE, nrows=2000)

print("Shape:", sample.shape)

print("\nColumns:")
print(sample.columns.tolist())

sample.head()

Filter to New Brunswick + Employment + Total Gender

In [ ]:
# Select only relevant columns to reduce memory usage
cols = [
    "REF_DATE",
    "GEO",
    "Labour force characteristics",
    "National Occupational Classification (NOC)",
    "Gender",
    "VALUE"
]

df = pd.read_csv(RAW_FILE, usecols=cols)

print("Full dataset shape:", df.shape)

df.head()

In [ ]:
# Filter for New Brunswick, Employment and Total Gender

df_nb = df[
    (df["GEO"] == "New Brunswick") &
    (df["Labour force characteristics"] == "Employment") &
    (df["Gender"] == "Total - Gender")
]

print("Filtered dataset shape:", df_nb.shape)

df_nb.head()

Select Occupation Level

In [ ]:
# Extract NOC code from the occupation label

df_nb["NOC_CODE"] = df_nb["National Occupational Classification (NOC)"].str.extract(r"\[(.*?)\]")

df_nb.head()

In [ ]:
# Create simplified sector groups

def classify_sector(noc):

    if pd.isna(noc):
        return "Other"

    noc = str(noc)

    if noc.startswith("21") or noc.startswith("22"):
        return "Tech_STEM"

    elif noc.startswith("11") or noc.startswith("12") or noc.startswith("13"):
        return "Business_Admin"

    elif noc.startswith("31") or noc.startswith("32"):
        return "Health"

    elif noc.startswith("62") or noc.startswith("63") or noc.startswith("64"):
        return "Sales_Service"

    else:
        return "Other"

df_nb["sector_group"] = df_nb["NOC_CODE"].apply(classify_sector)

df_nb.head()

Export Processed Dataset (Parquet + CSV sample)

In [ ]:
df_nb = df_nb[df_nb["REF_DATE"] >= 2000]

print("Dataset shape after year filter:", df_nb.shape)

In [ ]:
output_path = DATA_INTERIM / "nb_employment_by_occupation.parquet"

df_nb.to_parquet(output_path, index=False)

print("Dataset saved to:", output_path)

In [ ]:
df_nb.shape

In [ ]:
df_nb["sector_group"].value_counts()

Data Dictionary Notes

In [ ]:
# Data Dictionary for Processed Dataset (df_nb)

This dataset contains employment data for New Brunswick, filtered from the original Statistics Canada dataset. Below is a description of each column:

- **REF_DATE**: The reference year for the data (integer, e.g., 2000, 2001, etc.). Data is filtered to include only years from 2000 onwards.
- **GEO**: The geographic region (string). All records are for "New Brunswick".
- **Labour force characteristics**: The type of labour force metric (string). All records are for "Employment".
- **National Occupational Classification (NOC)**: The occupation description including the NOC code in brackets (string, e.g., "Total, all occupations [00-95]").
- **Gender**: The gender category (string). All records are for "Total - Gender" (combined genders).
- **VALUE**: The employment value in thousands of persons (float, e.g., 331.6).
- **NOC_CODE**: The extracted NOC code from the occupation description (string, e.g., "00-95"). Used for classification.
- **sector_group**: A simplified sector classification based on NOC_CODE (string, e.g., "Tech_STEM", "Health", "Other"). Classification logic:
    - Tech_STEM: NOC codes starting with 21 or 22.
    - Business_Admin: NOC codes starting with 11, 12, or 13.
    - Health: NOC codes starting with 31 or 32.
    - Sales_Service: NOC codes starting with 62, 63, or 64.
    - Other: All other codes or missing values.

**Notes**:
- The dataset has been filtered to New Brunswick, Employment, and Total Gender only.
- NOC_CODE is extracted using regex from the NOC column.
- sector_group is derived from NOC_CODE using the classify_sector function.
- Original data source: Statistics Canada (table 14-10-0416-01).
- Units: VALUE is in thousands of persons.
- Shape: As of the last cell, the dataset has 1534 rows and 8 columns.